# DynaPrompt Per-Token Analysis

**Setup:** Runtime → Change runtime type → T4 GPU → Save

In [ ]:
!nvidia-smi

## Clone Repository

In [ ]:
import os
os.chdir('/content')

!rm -rf 6694-DynaPrompt
!git clone https://github.com/ch3889/6694-DynaPrompt.git
os.chdir('/content/6694-DynaPrompt')
!git checkout zk2295

# Initialize submodules with verbose output
print('\nInitializing submodules...')
!git submodule update --init --recursive --progress

# Verify taming-transformers exists
print('\nVerifying submodules...')
!ls -la models/stable_diffusion_compvis/src/

# If taming-transformers still missing, clone it manually
import subprocess
if not os.path.exists('models/stable_diffusion_compvis/src/taming-transformers/taming'):
    print('\n⚠ Submodule failed, cloning taming-transformers manually...')
    os.makedirs('models/stable_diffusion_compvis/src', exist_ok=True)
    subprocess.run(['git', 'clone', 'https://github.com/CompVis/taming-transformers.git', 
                   'models/stable_diffusion_compvis/src/taming-transformers'], check=True)
    print('✓ taming-transformers cloned')

print(f'\nWorking directory: {os.getcwd()}')
!ls -la models/stable_diffusion_compvis/ldm/ | head -10

## Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q diffusers==0.21.4 accelerate safetensors huggingface-hub
!pip install -q omegaconf einops pytorch-lightning
!pip install -q Pillow numpy matplotlib tqdm scikit-image
!pip install -q kornia albumentations opencv-python imageio imageio-ffmpeg

## Install Transformers

In [ ]:
import subprocess, sys

try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.33.2'], check=True, timeout=120)
    print('✓ transformers')
except:
    print('⚠ Tokenizers build failed, installing without...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', '--no-deps'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'filelock', 'huggingface-hub', 'packaging', 'pyyaml', 'regex', 'requests', 'tqdm'])
    print('✓ transformers (no tokenizers)')

## Setup Python Paths

In [ ]:
import sys
import os

# First, check what's in taming-transformers
print('Checking taming-transformers structure...')
!ls -la /content/6694-DynaPrompt/models/stable_diffusion_compvis/src/taming-transformers/

# Add paths to Python's search path
sys.path.insert(0, '/content/6694-DynaPrompt')
sys.path.insert(0, '/content/6694-DynaPrompt/models/stable_diffusion_compvis')
sys.path.insert(0, '/content/6694-DynaPrompt/models/stable_diffusion_compvis/src/clip')

# Fix taming module - check if taming folder exists inside taming-transformers
taming_base = '/content/6694-DynaPrompt/models/stable_diffusion_compvis/src/taming-transformers'
if os.path.exists(os.path.join(taming_base, 'taming')):
    # taming folder exists, add parent to path
    sys.path.insert(0, taming_base)
    print(f'✓ Added taming from: {taming_base}')
else:
    # taming folder doesn't exist, the whole folder IS taming
    # Create a symlink
    import subprocess
    subprocess.run(['ln', '-sf', taming_base, '/content/taming'], check=False)
    sys.path.insert(0, '/content')
    print(f'✓ Created taming symlink')

# Fix pytorch_lightning compatibility
try:
    from pytorch_lightning.utilities.distributed import rank_zero_only
except ImportError:
    import pytorch_lightning as pl
    from types import ModuleType
    
    if not hasattr(pl, 'utilities'):
        pl.utilities = ModuleType('utilities')
        sys.modules['pytorch_lightning.utilities'] = pl.utilities
    
    if not hasattr(pl.utilities, 'distributed'):
        pl.utilities.distributed = ModuleType('distributed')
        sys.modules['pytorch_lightning.utilities.distributed'] = pl.utilities.distributed
    
    try:
        from pytorch_lightning.utilities.rank_zero import rank_zero_only
        pl.utilities.distributed.rank_zero_only = rank_zero_only
    except ImportError:
        def rank_zero_only(fn):
            return fn
        pl.utilities.distributed.rank_zero_only = rank_zero_only
    
    print('✓ PyTorch Lightning compatibility')

# Test imports
try:
    import ldm
    print('✓ ldm module')
except ImportError as e:
    print(f'❌ ldm: {e}')

try:
    import taming
    print(f'✓ taming module (from {taming.__file__})')
except ImportError as e:
    print(f'❌ taming: {e}')
    print('\nDebugging: sys.path entries with taming:')
    for p in sys.path:
        if 'taming' in p.lower():
            print(f'  {p}')


## Load CLIP

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch

print('Loading CLIP...')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
print('✓ CLIP ready')

## Download Stable Diffusion v1.5

In [ ]:
from huggingface_hub import hf_hub_download

print('Downloading Stable Diffusion v1.5...')
checkpoint_path = hf_hub_download(
    repo_id='runwayml/stable-diffusion-v1-5',
    filename='v1-5-pruned-emaonly.ckpt',
    cache_dir='/content/models'
)
print(f'✓ Downloaded to: {checkpoint_path}')

## Initialize DynaPrompt

In [ ]:
from dynaprompt.wrapper import DynaPromptPipeline
import torch

print('Initializing DynaPrompt with per-token analysis...')
dynaprompt = DynaPromptPipeline(
    config_path='/content/6694-DynaPrompt/configs/dynaprompt_config.yaml',
    ckpt_path=checkpoint_path,
    device='cuda'
)
print('✓ DynaPrompt ready!')

## Generate Test Image

In [ ]:
prompt = 'a golden retriever playing with a red ball in a snowy park'
print(f'Prompt: "{prompt}"')
print('\nGenerating image (2-3 minutes)...')

results = dynaprompt.generate_with_feedback(
    prompt=prompt,
    steps=50,
    cfg_scale=7.5,
    height=512,
    width=512,
    seed=42,
    feedback_enabled=True
)

from PIL import Image
import numpy as np

img = results['images'][0]
if isinstance(img, torch.Tensor):
    img = img.cpu().numpy()
if img.max() <= 1.0:
    img = (img * 255).astype(np.uint8)
    
print('\n✓ Generation complete!')
Image.fromarray(img)

## Baseline Comparison

In [ ]:
import matplotlib.pyplot as plt

print('Generating baseline (WITHOUT feedback)...')
results_baseline = dynaprompt.generate_with_feedback(
    prompt=prompt,
    steps=50,
    cfg_scale=7.5,
    height=512,
    width=512,
    seed=42,
    feedback_enabled=False
)

img_baseline = results_baseline['images'][0]
if isinstance(img_baseline, torch.Tensor):
    img_baseline = img_baseline.cpu().numpy()
if img_baseline.max() <= 1.0:
    img_baseline = (img_baseline * 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(20, 10))
axes[0].imshow(img_baseline)
axes[0].set_title('WITHOUT DynaPrompt Feedback', fontsize=16, fontweight='bold', color='red')
axes[0].axis('off')

axes[1].imshow(img)
axes[1].set_title('WITH DynaPrompt Feedback', fontsize=16, fontweight='bold', color='green')
axes[1].axis('off')

plt.tight_layout()
plt.show()
print('\n✓ Comparison complete!')